# Topic: SQL Recursive Query Pattern (WITH RECURSIVE)

## Definition (30-second explanation)
* A Recursive Common Table Expression (CTE) is a query that references itself. 
* It processes hierarchical or iterative data by starting with a non-recursive base case (anchor query) and repeatedly joining it to itself.
* The iteration continues until a specific termination condition is met.

## Why Interviewers Ask This
* It demonstrates advanced SQL mastery and the ability to replace procedural code (like loops) with declarative logic.
* It proves you can navigate complex data structures like organizational trees, bill-of-materials, or directed graphs.
* It tests your attention to detail regarding query safety, specifically your ability to prevent infinite loops.

## Core Concepts
* **Anchor Member:** The starting row(s) that run exactly once to provide the initial result set.
* **UNION ALL:** The mandatory set operator connecting the anchor to the recursive step (using standard UNION causes errors).
* **Recursive Member:** The query that joins back to the CTE itself to fetch the next level of data.
* **Termination Condition:** A filter (e.g., `WHERE depth < N`) that ensures the recursion stops.

## When to Use
* **Organizational Hierarchies:** Finding all direct and indirect reports for a manager.
* **Date Sequence Generation:** Generating a calendar table for a specific year to fill gaps in time-series data.
* **Bill-of-Materials (BOM):** Calculating total raw materials needed for a parent product.
* **Graph Traversal:** Finding paths between nodes in a network.

## Advantages
* Efficiently handles hierarchical data of unknown depths in a single query.
* Dynamically generates missing records (like date gaps) without creating physical tables.
* Keeps logic inside the database, reducing the amount of data transferred to application layers.

## Limitations
* High risk of infinite loops if cyclical data (A reports to B, B reports to A) is not handled or a depth limit is forgotten.
* Aggregate functions (GROUP BY, HAVING), DISTINCT, and certain joins are strictly forbidden inside the recursive member.
* Performance can degrade rapidly on very wide or deep hierarchies without proper indexing on join columns.

## Common Comparisons
* **Recursive CTEs vs. Procedural Loops (WHILE):** CTEs are declarative, better optimized by query engines, and often read cleaner than cursor-based procedural loops.
* **Recursive CTEs vs. Self-Joins:** Self-joins are only effective when the hierarchy depth is fixed and known (e.g., exactly 3 levels). CTEs handle dynamic, unknown depths.

## Common Interview Traps
* **Missing a stopping condition:** Forgetting `WHERE level < N` can lock up or crash the query.
* **Using `UNION` instead of `UNION ALL`:** Standard `UNION` attempts to remove duplicates, which violates SQL recursion rules.
* **Syntax Dialect Issues:** Writing `WITH RECURSIVE` in SQL Server/Oracle (which only use `WITH`) or forgetting the `RECURSIVE` keyword in MySQL/PostgreSQL.

## Python / SQL Syntax
```sql
WITH RECURSIVE cte_name AS (
    -- ANCHOR MEMBER (base case)
    SELECT id, name, 1 AS level 
    FROM hierarchy_table 
    WHERE parent_id IS NULL
    
    UNION ALL 
    
    -- RECURSIVE MEMBER
    SELECT h.id, h.name, c.level + 1
    FROM hierarchy_table h
    JOIN cte_name c ON h.parent_id = c.id
    WHERE c.level < 10 -- Mandatory safety limit
)
SELECT * FROM cte_name;
```

## 45-Second Interview Answer
"A recursive CTE is an advanced SQL pattern used to iterate over hierarchical data, like org charts, or to generate dynamic sequences like date calendars. It consists of an anchor member that runs once to establish a base case, joined via a UNION ALL to a recursive member that repeatedly joins back to the CTE. The recursion stops when no new rows are produced. In an interview, I always ensure I use UNION ALL instead of UNION, and I enforce a termination condition, like a maximum depth, to guarantee I don't create an infinite loop."

## Example Questions:

### Q1: Given an org chart table (emp_id, emp_name, manager_id), find all direct and indirect reports of a specific manager (e.g., Vikram VP IT).

**Ideal Interview Answer (MySQL):**
```sql
WITH RECURSIVE Subordinates AS (
    -- Anchor: Start with Vikram
    SELECT emp_id, emp_name, manager_id, 1 AS depth
    FROM employees
    WHERE emp_name = 'Vikram (VP IT)'
    
    UNION ALL
    
    -- Recursive: Find everyone reporting to the people in the CTE
    SELECT e.emp_id, e.emp_name, e.manager_id, s.depth + 1
    FROM employees e
    JOIN Subordinates s ON e.manager_id = s.emp_id
    WHERE s.depth < 15 -- safety limit
)
SELECT * FROM Subordinates;
```
* **Common Mistake:** Starting the anchor with `manager_id IS NULL`. The question specifically asks for a *specific* manager's reports, so the anchor must filter for that specific employee.
* **Likely Follow-up:** "How would you modify this to also output the hierarchical path (e.g., Vikram -> Rahul)?"

### Q2: Generate a sequence of numbers from 1 to 100 using a recursive CTE. Then use it to create a multiplication table.

**Ideal Interview Answer (MySQL):**
```sql
WITH RECURSIVE numbers AS (
    SELECT 1 AS n
    UNION ALL
    SELECT n + 1 
    FROM numbers 
    WHERE n < 100
)
-- Multiplication table example (e.g., 5 times table)
SELECT 
    5 AS base_number,
    n AS multiplier,
    (5 * n) AS result
FROM numbers
WHERE n <= 10;
```
* **Common Mistake:** Using `UNION` instead of `UNION ALL`, or writing `WHERE n <= 100` in the recursive step which actually generates 101 rows (since when n=100, it passes the check and adds 101).
* **Likely Follow-up:** "Why is a recursive CTE better than creating a physical 'numbers' table in your database for this task?"

### Q3: Find the shortest path between two nodes in a graph using a recursive CTE with a visited nodes tracker.

**Ideal Interview Answer (MySQL):**
```sql
WITH RECURSIVE GraphPath AS (
    -- Anchor: Start at Node A
    SELECT 
        source, 
        target, 
        distance, 
        CAST(source AS CHAR(200)) AS path,
        1 AS stops
    FROM edges
    WHERE source = 'A'
    
    UNION ALL
    
    -- Recursive: Traverse edges
    SELECT 
        e.source, 
        e.target, 
        p.distance + e.distance,
        CONCAT(p.path, '->', e.target),
        p.stops + 1
    FROM edges e
    JOIN GraphPath p ON e.source = p.target
    -- Visited nodes tracker to prevent infinite loops in cyclic graphs
    WHERE FIND_IN_SET(e.target, REPLACE(p.path, '->', ',')) = 0
      AND p.stops < 10
)
SELECT path, distance 
FROM GraphPath 
WHERE target = 'Z' 
ORDER BY distance ASC 
LIMIT 1;
```
* **Common Mistake:** Failing to track visited nodes. Graph data is inherently cyclical. Without `FIND_IN_SET` or string concatenation checks, the query will loop infinitely.
* **Likely Follow-up:** "Is SQL the best tool for calculating shortest paths on massive graphs? What are the alternatives?"

### Q4: Given a product bill-of-materials table (product_id, component_id, quantity), calculate the total quantity of each raw material needed to produce 100 units of the final product.

**Ideal Interview Answer (MySQL):**
```sql
WITH RECURSIVE BOM_Explosion AS (
    -- Anchor: Start with the final product (e.g., Product 1)
    SELECT 
        component_id, 
        quantity * 100 AS total_qty
    FROM bill_of_materials
    WHERE product_id = 1
    
    UNION ALL
    
    -- Recursive: Multiply parent quantities by child quantities
    SELECT 
        b.component_id, 
        b.quantity * p.total_qty AS total_qty
    FROM bill_of_materials b
    JOIN BOM_Explosion p ON b.product_id = p.component_id
)
-- Aggregate final raw materials (components that don't have further sub-components)
SELECT 
    component_id, 
    SUM(total_qty) AS required_quantity
FROM BOM_Explosion
GROUP BY component_id;
```
* **Common Mistake:** Trying to use `SUM()` or `GROUP BY` *inside* the recursive CTE member. Aggregations must be done in the final `SELECT` outside the CTE.
* **Likely Follow-up:** "How does the query optimizer handle execution of this recursive CTE underneath the hood?"